In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

BASE_PATH = "/content/drive/MyDrive/Tracefinder"

print("Exists:", os.path.exists(BASE_PATH))
print("Contents:", os.listdir(BASE_PATH) if os.path.exists(BASE_PATH) else "Not found")


Exists: True
Contents: ['Flatfield', 'Originals', 'Official', 'Wikipedia', 'Tampered images', 'tracefinder_manifest_preprocessed.csv', 'tracefinder_manifest.csv', 'Results', 'official_preprocessing.csv', 'flatfield_preprocessing.csv', 'wikipedia_preprocessing.csv', 'signature_summary.csv', 'signature_pairwise_distances.csv', 'scanner_signatures.csv', 'official_exploration.csv', 'wikipedia_exploration.csv', 'flatfield_index.csv', 'official_index.csv', 'wikipedia_index.csv', 'features', 'dataset_labels.csv', 'processed_data', 'models', 'results', 'processed', 'patches', 'cnn_data', 'cnn_temp']


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

CNN_ROOT = "/content/drive/MyDrive/Tracefinder/cnn_temp"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.3   # 70% train, 30% test+val
)

train_data = datagen.flow_from_directory(
    CNN_ROOT,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

val_data = datagen.flow_from_directory(
    CNN_ROOT,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

NUM_CLASSES = train_data.num_classes

print("Train samples:", train_data.samples)
print("Validation/Test samples:", val_data.samples)
print("Classes:", train_data.class_indices)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[early_stop]
)

# Save model (IMPORTANT)
model.save("cnn_scanner_model.keras")
print("✅ CNN model saved")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Predict on validation/test data
y_pred = model.predict(val_data)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = val_data.classes

# Accuracy
acc = np.mean(y_pred_classes == y_true)
print("CNN Accuracy:", acc)

# Classification report
print("\nCNN Classification Report:\n")
print(classification_report(
    y_true,
    y_pred_classes,
    target_names=val_data.class_indices.keys()
))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10,8))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=val_data.class_indices.keys(),
    yticklabels=val_data.class_indices.keys()
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("CNN Confusion Matrix")
plt.show()

# Accuracy / Loss Curves
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.legend()
plt.title("Accuracy")

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.legend()
plt.title("Loss")

plt.show()


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

CNN_ROOT = "/content/drive/MyDrive/Tracefinder/cnn_temp"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# We use ONLY validation split for testing the loaded model
test_data = datagen.flow_from_directory(
    CNN_ROOT,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print("Test samples:", test_data.samples)
print("Classes:", test_data.class_indices)


Found 911 images belonging to 11 classes.
Test samples: 911
Classes: {'Canon120-1': 0, 'Canon120-2': 1, 'Canon220': 2, 'Canon9000-1': 3, 'Canon9000-2': 4, 'EpsonV370-1': 5, 'EpsonV370-2': 6, 'EpsonV39-1': 7, 'EpsonV39-2': 8, 'EpsonV550': 9, 'HP': 10}


In [6]:
from tensorflow.keras.models import load_model

MODEL_PATH = "/content/cnn_model.h5"


model = load_model(MODEL_PATH)

model.summary()
print("✅ Model loaded successfully")


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       802,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 11)             │           715 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 822,413 (3.14 MB)

 Trainable params: 822,411 (3.14 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

✅ Model loaded successfully


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Predict
y_pred = model.predict(test_data)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_data.classes

# Accuracy
accuracy = (y_pred_classes == y_true).mean()
print("CNN Accuracy:", accuracy)

# Classification report
print("\nCNN Classification Report:\n")
print(classification_report(
    y_true,
    y_pred_classes,
    target_names=test_data.class_indices.keys()
))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10,8))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=test_data.class_indices.keys(),
    yticklabels=test_data.class_indices.keys()
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("CNN Confusion Matrix (Loaded Model)")
plt.show()
